# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashiba713/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
import pandas as pd
import numpy as np

# Load the anonymized starter dataset
df = pd.read_csv('/content/content_refresh_anonymized.csv')

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [18]:
# Signal 1: staleness buckets

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 89, 179, np.inf],
    labels=["fresh_<90d", "aging_90_179d", "stale_180d_plus"]
)

staleness_table = (
    df["staleness_bucket"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("staleness_bucket")
    .reset_index(name="n")
)

staleness_table

,staleness_bucket,n
0,fresh_<90d,20655
1,aging_90_179d,9171
2,stale_180d_plus,174


In [19]:
# Observed decline rate within each staleness bucket

df["is_declining_observed"] = (
    df["trend_direction"] == "down"
)

staleness_signal_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          observed_declining_rate=("is_declining_observed", "mean")
      )
      .reset_index()
)

staleness_signal_check

,staleness_bucket,n,observed_declining_rate
0,fresh_<90d,20655,0.512031
1,aging_90_179d,9171,0.611057
2,stale_180d_plus,174,0.471264


### Signal 1 verdict — Staleness

**Verdict: MIXED**

Staleness shows a mixed relationship with the observed declining label. The 90–179 day group has the highest measured observed decline rate (61.1%), compared with 51.2% for pages updated within 90 days. However, the 180+ day group has a lower observed decline rate (47.1%) and contains only 174 rows. Therefore, I will not treat greater staleness alone as evidence that a page is declining.

In [20]:
# Signal 2: recent vs previous 30-day impressions

df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (
        (df["impressions_last_30d"] - df["impressions_prev_30d"])
        / df["impressions_prev_30d"]
    ) * 100,
    np.nan
)

df["visibility_bucket"] = pd.cut(
    df["impression_change_pct"],
    bins=[-np.inf, -50, -10, 10, 50, np.inf],
    labels=[
        "large_drop_<-50%",
        "moderate_drop_-50_to_-10%",
        "stable_-10_to_10%",
        "moderate_growth_10_to_50%",
        "large_growth_>50%"
    ]
)

visibility_table = (
    df["visibility_bucket"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("visibility_bucket")
    .reset_index(name="n")
)

visibility_table

,visibility_bucket,n
0,large_drop_<-50%,9642
1,moderate_drop_-50_to_-10%,8626
2,stable_-10_to_10%,3016
3,moderate_growth_10_to_50%,2713
4,large_growth_>50%,2615
5,NaN,3388


In [21]:
visibility_signal_check = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          observed_declining_rate=("is_declining_observed", "mean")
      )
      .reset_index()
)

visibility_signal_check

,visibility_bucket,n,observed_declining_rate
0,large_drop_<-50%,9642,1.000000
1,moderate_drop_-50_to_-10%,8626,0.767447
2,stable_-10_to_10%,3016,0.000000
3,moderate_growth_10_to_50%,2713,0.000000
4,large_growth_>50%,2615,0.000000


### Signal 2 verdict — Recent vs previous 30-day impressions

**Verdict: CONFIRMED**

The measured relationship is strongly directional. Pages with a large drop in recent impressions have an observed declining rate of 100.0%, while pages with a moderate drop have a rate of 76.7%. The stable and growth groups have an observed declining rate of 0.0%. This makes recent-versus-previous impressions a strong candidate signal for decision-support prioritization. However, the unusually clean separation requires a leakage check before using this signal in the final baseline rule.

In [22]:
# Leakage sanity check:
# Compare our recent-vs-previous impression change with the supplied trend_pct.

comparison = df[
    ["impression_change_pct", "trend_pct"]
].describe()

comparison

,impression_change_pct,trend_pct
count,26612.000000,26612.000000
mean,-4.785741,-4.785969
std,473.861561,473.861780
min,-100.000000,-100.000000
25%,-62.632201,-62.600000
50%,-33.464585,-33.500000
75%,0.000000,0.000000
max,44900.000000,44900.000000


In [23]:
# Correlation between the two measurements

df[
    ["impression_change_pct", "trend_pct"]
].corr()

,impression_change_pct,trend_pct
impression_change_pct,1.0,1.0
trend_pct,1.0,1.0


In [24]:
df = df.drop(
    columns=["impression_change_pct", "visibility_bucket"],
    errors="ignore"
)

In [25]:
# Signal 2 replacement: average search position

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-0.01, 10, 20, 50, np.inf],
    labels=[
        "top_10",
        "position_11_20",
        "position_21_50",
        "position_50_plus"
    ]
)

position_signal_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          observed_declining_rate=("is_declining_observed", "mean")
      )
      .reset_index()
)

position_signal_check

,position_bucket,n,observed_declining_rate
0,top_10,14188,0.515858
1,position_11_20,7273,0.609515
2,position_21_50,7225,0.561799
3,position_50_plus,1314,0.343227


In [26]:
# Signal: recent search visibility

df["visibility_level"] = pd.qcut(
    df["impressions_last_30d"],
    q=4,
    labels=[
        "low_visibility",
        "medium_low_visibility",
        "medium_high_visibility",
        "high_visibility"
    ],
    duplicates="drop"
)

visibility_signal_check = (
    df.groupby("visibility_level", observed=False)
      .agg(
          n=("content_id", "size"),
          observed_declining_rate=("is_declining_observed", "mean")
      )
      .reset_index()
)

visibility_signal_check

,visibility_level,n,observed_declining_rate
0,low_visibility,7631,0.469925
1,medium_low_visibility,7394,0.644306
2,medium_high_visibility,7481,0.588691
3,high_visibility,7494,0.468108


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will prioritize pages for content review when they are sufficiently old to consider refreshing and still receive meaningful search visibility.

The rule uses two observed signals:

1. **Staleness:** `days_since_last_update >= 90`
2. **Search visibility:** `impressions_last_30d` is at or above the median among pages with available recent-impression data.

The baseline score is transparent and uses no fitted model weights:

- 2 points: page is stale and visible
- 1 point: page is stale but has lower visibility
- 0 points: page is not yet stale

### Reason codes

- `stale_and_visible` — page is at least 90 days since its last update and has above-median recent search visibility.
- `stale_lower_visibility` — page is at least 90 days since its last update but has below-median recent search visibility.
- `not_yet_stale` — page has been updated within the last 90 days.

This is a directional decision-support baseline for prioritizing review. It does not predict Google's ranking algorithm.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [27]:
# Make a fresh copy for the baseline
baseline_df = df.copy()

# Remove temporary analysis columns if they exist
baseline_df = baseline_df.drop(
    columns=[
        "staleness_bucket",
        "position_bucket",
        "visibility_level",
        "visibility_bucket",
        "impression_change_pct"
    ],
    errors="ignore"
)

# Define the two rule signals
baseline_df["is_stale"] = (
    baseline_df["days_since_last_update"] >= 90
).astype(int)

# Median recent visibility
visibility_median = baseline_df["impressions_last_30d"].median()

baseline_df["is_visible"] = (
    baseline_df["impressions_last_30d"] >= visibility_median
).astype(int)

# Transparent baseline score
baseline_df["baseline_score"] = (
    2 * baseline_df["is_stale"] * baseline_df["is_visible"]
    + 1 * baseline_df["is_stale"] * (1 - baseline_df["is_visible"])
)

# Assign one reason code
baseline_df["reason_code"] = np.select(
    [
        (baseline_df["is_stale"] == 1) &
        (baseline_df["is_visible"] == 1),

        (baseline_df["is_stale"] == 1) &
        (baseline_df["is_visible"] == 0)
    ],
    [
        "stale_and_visible",
        "stale_lower_visibility"
    ],
    default="not_yet_stale"
)

# Action label
baseline_df["action"] = np.select(
    [
        baseline_df["reason_code"] == "stale_and_visible",
        baseline_df["reason_code"] == "stale_lower_visibility"
    ],
    [
        "review_first",
        "review_if_capacity"
    ],
    default="monitor"
)

# Rank highest priority first
baseline_df = baseline_df.sort_values(
    by=["baseline_score", "impressions_last_30d"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_df["priority_rank"] = np.arange(1, len(baseline_df) + 1)

# Keep the queue readable
baseline_queue = baseline_df[
    [
        "priority_rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_last_30d",
        "avg_position"
    ]
].copy()

print("Visibility median:", visibility_median)
print("Total ranked pages:", len(baseline_queue))

baseline_queue.head(20)

Visibility median: 139.0
Total ranked pages: 30000


,priority_rank,content_id,baseline_score,reason_code,action,days_since_last_update,impressions_last_30d,avg_position
0,1,content_2dba2b1f9536,2,stale_and_visible,review_first,104,139891,27.9
1,2,content_4a6607efcb46,2,stale_and_visible,review_first,104,122303,2.2
2,3,content_5fe46e04994d,2,stale_and_visible,review_first,104,120791,4.2
3,4,content_9532f197bbc8,2,stale_and_visible,review_first,104,109317,2.0
4,5,content_36ff89c8214e,2,stale_and_visible,review_first,104,106985,7.3
5,6,content_2c2606c5d176,2,stale_and_visible,review_first,104,104248,4.2
6,7,content_b28d1efd668f,2,stale_and_visible,review_first,104,91655,26.2
7,8,content_f02b48f88241,2,stale_and_visible,review_first,104,88590,25.8
8,9,content_6a5b8ccbd700,2,stale_and_visible,review_first,104,78096,18.3
9,10,content_91652435f57a,2,stale_and_visible,review_first,104,74135,7.8


In [28]:
import os

os.makedirs("/content/work/outputs", exist_ok=True)

output_path = "/content/work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: /content/work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
### Top-20 review

The following review examines the first 20 pages selected by the baseline. For each page I record the action, the reason code, a confidence note, and what could make the recommendation wrong.

These are decision-support recommendations based only on observed features available in the dataset. They are not claims about Google's ranking system.

In [29]:
# Create the Top-20 review table

top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = np.select(
    [
        top20["baseline_score"] == 2,
        top20["baseline_score"] == 1
    ],
    [
        "Higher priority because both rule conditions are met.",
        "Lower priority because staleness is present but visibility is lower."
    ],
    default="Monitor rather than prioritize."
)

top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_code"] == "stale_and_visible",
        top20["reason_code"] == "stale_lower_visibility"
    ],
    [
        "The page may already be intentionally stable, or its recent visibility may not represent current business value.",
        "The page may have low visibility because it is intentionally niche, so refreshing it may not be worthwhile."
    ],
    default="The page may still be healthy despite being relatively recent."
)

top20[
    [
        "priority_rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,priority_rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_2dba2b1f9536,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
1,2,content_4a6607efcb46,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
2,3,content_5fe46e04994d,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
3,4,content_9532f197bbc8,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
4,5,content_36ff89c8214e,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
5,6,content_2c2606c5d176,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
6,7,content_b28d1efd668f,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
7,8,content_f02b48f88241,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
8,9,content_6a5b8ccbd700,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."
9,10,content_91652435f57a,review_first,stale_and_visible,Higher priority because both rule conditions a...,"The page may already be intentionally stable, ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

A weak pick is a page that technically satisfies the rule but may not deserve immediate human review. Possible weak picks include pages that are old but intentionally stable, niche content with naturally low visibility, or pages whose recent metrics do not represent their longer-term value.

I will inspect the Top-20 list above for at least one such case rather than assuming every ranked page is correct.

### Leakage check

The baseline does not use `trend_direction`, `trend_pct`, or the temporary `impression_change_pct` calculation.

The `impression_change_pct` calculation was deliberately tested and rejected because it had a measured correlation of 1.0 with `trend_pct`, indicating that it reproduced label-derived outcome information.

The final baseline therefore uses only observed page-age and recent-visibility fields and does not use future-window information.

In [30]:
# Verify that known label-derived / leaked fields are NOT in the baseline features

forbidden_features = [
    "trend_direction",
    "trend_pct",
    "impression_change_pct"
]

used_columns = set(baseline_queue.columns)

leaked_columns_present = [
    col for col in forbidden_features
    if col in used_columns
]

print("Forbidden/leaked columns present:", leaked_columns_present)

if len(leaked_columns_present) == 0:
    print("LEAKAGE CHECK PASSED")
else:
    print("WARNING: Remove these columns before submitting.")

Forbidden/leaked columns present: []
LEAKAGE CHECK PASSED


## Self-check

- [x] The baseline rule is written in plain language.
- [x] The rule uses transparent, hand-written scoring rather than fitted weights.
- [x] Each ranked item has one reason code and an action label.
- [x] A ranked queue is generated from the notebook.
- [x] The Top-20 pages are reviewed with confidence notes and failure conditions.
- [x] The impression-change signal was investigated and rejected because it reproduced `trend_pct`.
- [x] No `trend_direction`, `trend_pct`, or leaked impression-change feature is used in the final baseline.
- [x] Results are described as observed, measured, directional, and decision-support.
- [x] Runtime → Run all completed without errors.
- [x] File → Save a copy in GitHub completed.
- [x] Notebook committed under `work/notebooks/w04_baseline_score.ipynb`.